# Cliente API de OntoPortal

Este notebook sirve como un cliente interactivo para explorar la API de OntoPortal. OntoPortal es una plataforma que proporciona acceso a ontologías biomédicas y permite realizar búsquedas, anotaciones y análisis de términos y conceptos.

En este notebook, aprenderemos cómo conectarnos a la API y realizaremos diversas llamadas para demostrar las capacidades del servicio.

## 1. Importar Librerías Requeridas

Para interactuar con la API de OntoPortal, utilizaremos la librería `requests` para realizar llamadas HTTP y `json` para manejar las respuestas. También importaremos `pandas` para manipular datos si es necesario.

In [40]:
import requests
import json
import pandas as pd
from pprint import pprint
from urllib.parse import urlparse, quote

## 2. Configurar la Conexión a la API

Para conectarnos a la API de OntoPortal, necesitamos definir la URL base del servicio. Cuando OntoPortal se despliega con Docker Compose, la API está disponible a través de Nginx en `http://localhost/api`. 

La API de OntoPortal requiere una API key para autenticación, que se obtiene del archivo `.env` del proyecto. Esta key se incluye automáticamente en todas las solicitudes.

In [42]:
# Configurar la URL base de la API

## Local
#BASE_URL = "http://localhost/api"
#API_KEY = "6410be68-e26a-4dc3-a316-d821c64e9108"  # API Key obtenida del archivo .env

## Remoto
BASE_URL = "https://161.111.18.191/api"
API_KEY="da67ebc7-5006-4a61-b8b9-69cca3fcf208"


# Función auxiliar para hacer requests
def api_request(endpoint, params=None):
    if params is None:
        params = {}
    params['apikey'] = API_KEY  # Agregar API key a todos los requests
    url = f"{BASE_URL}{endpoint}"
    response = requests.get(url, params=params)
    response.raise_for_status()
    return response.json()

print("Conexión configurada. URL base:", BASE_URL)

Conexión configurada. URL base: https://161.111.18.191/api


## 3. Obtener Listado de Ontologías Disponibles

La API permite obtener un listado de todas las ontologías disponibles en el portal. Esto es útil para explorar qué vocabularios están cargados en el sistema.

In [43]:
# Obtener listado de ontologías
ontologies = api_request("/ontologies")

print(f"Se encontraron {len(ontologies)} ontologías:")
for ont in ontologies[:5]:  # Mostrar las primeras 5
    print(f"- {ont.get('acronym', 'N/A')}: {ont.get('name', 'Sin nombre')}")

# Mostrar en formato tabla si hay pandas
if ontologies:
    df = pd.DataFrame(ontologies)
    # Seleccionar solo las columnas que existen
    available_columns = [col for col in ['acronym', 'name', 'description'] if col in df.columns]
    if available_columns:
        display(df[available_columns].head())
    else:
        print("No se encontraron las columnas esperadas en los datos.")

SSLError: HTTPSConnectionPool(host='161.111.18.191', port=443): Max retries exceeded with url: /api/ontologies?apikey=da67ebc7-5006-4a61-b8b9-69cca3fcf208 (Caused by SSLError(SSLCertVerificationError(1, '[SSL: CERTIFICATE_VERIFY_FAILED] certificate verify failed: self-signed certificate (_ssl.c:1129)')))

## 4. Obtener Metadatos de una Ontología

Para obtener detalles específicos de una ontología, podemos usar su acrónimo. El endpoint principal `GET /ontologies/{acronym}` devuelve los metadatos generales, pero la información más completa del vocabulario está en la última `submission` y en sus `metrics`.

In [35]:
# Obtener metadatos de una ontología específica
if ontologies:
    acronym = ontologies[0]['acronym']
    ontology_details = api_request(f"/ontologies/{acronym}")
    latest_submission = api_request(f"/ontologies/{acronym}/latest_submission")
    submission_metrics = None
    if latest_submission and latest_submission.get('links') and latest_submission['links'].get('metrics'):
        metrics_url = latest_submission['links']['metrics']
        if metrics_url.startswith('http'):
            parsed = urlparse(metrics_url)
            metrics_path = parsed.path
        else:
            metrics_path = metrics_url if metrics_url.startswith('/') else f'/{metrics_url}'
        submission_metrics = api_request(metrics_path)

    print(f"Detalles de la ontología '{acronym}':")
    display({
        'Acrónimo': ontology_details.get('acronym'),
        'Nombre': ontology_details.get('name'),
        'Tipo': ontology_details.get('ontologyType'),
        'ID': ontology_details.get('@id'),
    })

    print("\nInformación de la última submission:")
    submission_summary = {
        'Descripción': latest_submission.get('description'),
        'Estado': latest_submission.get('status'),
        'Formato': latest_submission.get('hasOntologyLanguage'),
        'Fecha de creación': latest_submission.get('creationDate'),
        'Fecha de publicación': latest_submission.get('released'),
        'Versión': latest_submission.get('version'),
        'Homepage': latest_submission.get('homepage'),
        'Documentación': latest_submission.get('documentation'),
        'Contacto': ', '.join([c.get('email', c.get('name', '')) for c in latest_submission.get('contact', [])]) if latest_submission.get('contact') else None,
    }
    display(submission_summary)

    print("\nMétricas de la última submission:")
    if submission_metrics:
        metrics_summary = {
            'Términos': submission_metrics.get('ontolexEntries'),
            'Formas': submission_metrics.get('ontolexForms'),
            'Sentidos léxicos': submission_metrics.get('ontolexSenses'),
            'Conceptos léxicos': submission_metrics.get('ontolexConcepts'),
            'Clases': submission_metrics.get('classes'),
            'Propiedades': submission_metrics.get('properties'),
            'Individuos': submission_metrics.get('individuals'),
            'Profundidad máxima': submission_metrics.get('maxDepth'),
            'Máximo hijos': submission_metrics.get('maxChildCount'),
        }
        display(metrics_summary)
    else:
        print("No se pudieron obtener métricas para la última submission.")
else:
    print("No hay ontologías disponibles para mostrar detalles.")

Detalles de la ontología 'DM':


{'Acrónimo': 'DM',
 'Nombre': 'Dispositivos Móviles',
 'Tipo': 'http://api:9393/ontology_types/ONTOLOGY',
 'ID': 'http://api:9393/ontologies/DM'}


Información de la última submission:


{'Descripción': 'Descripción de la terminología',
 'Estado': 'production',
 'Formato': 'ONTOLEX',
 'Fecha de creación': '2026-03-23T00:00:00+00:00',
 'Fecha de publicación': '2026-03-23T00:00:00+00:00',
 'Versión': None,
 'Homepage': None,
 'Documentación': None,
 'Contacto': 'carlos.badenes@upm.es'}


Métricas de la última submission:


{'Términos': 651,
 'Formas': 653,
 'Sentidos léxicos': 651,
 'Conceptos léxicos': 104,
 'Clases': 0,
 'Propiedades': 0,
 'Individuos': 0,
 'Profundidad máxima': 0,
 'Máximo hijos': 0}

## 5. Buscar Conceptos en una Ontología

La API permite buscar conceptos dentro de una ontología específica usando términos de búsqueda. Para vocabularios como `DM`, los datos son léxicos, por lo que la búsqueda debe realizarse sobre `terminological_entries` para obtener entradas con sus sentidos y relaciones.

In [36]:
# Buscar términos léxicos en una ontología
if ontologies:
    acronym = ontologies[0]['acronym']
    search_term = "codi"  # Use un término de búsqueda sin acento cuando la búsqueda exacta no devuelve resultados

    def encode_id_for_api(uri):
        if isinstance(uri, dict):
            uri = uri.get('uri') or uri.get('@id') or str(uri)
        return quote(str(uri), safe='')

    def get_terminological_entry(acronym, entry_id):
        encoded_id = encode_id_for_api(entry_id)
        return api_request(f"/ontologies/{acronym}/terminological_entries/{encoded_id}")

    def pretty_entry_label(entry):
        if not entry:
            return None
        if entry.get('writtenReps'):
            if isinstance(entry['writtenReps'], list):
                return "; ".join([str(x) if isinstance(x, str) else x.get('uri', str(x)) for x in entry['writtenReps']])
            return str(entry['writtenReps'])
        if entry.get('form'):
            forms = entry['form']
            if isinstance(forms, list):
                form_strs = []
                for f in forms:
                    if isinstance(f, dict):
                        form_strs.append(f.get('uri') or f.get('@id') or str(f))
                    else:
                        form_strs.append(str(f))
                return "; ".join(form_strs)
            return str(forms)
        entry_id = entry.get('@id') or entry.get('id')
        if isinstance(entry_id, dict):
            entry_id = entry_id.get('uri') or entry_id.get('@id') or str(entry_id)
        return str(entry_id)

    def print_lexical_entry(entry):
        entry_id = entry.get('@id') or entry.get('id')
        if isinstance(entry_id, dict):
            entry_id = entry_id.get('uri') or entry_id.get('@id') or str(entry_id)
        print("ID:", entry_id)
        print("Texto:", pretty_entry_label(entry))
        if entry.get('language'):
            print("Idioma:", entry.get('language'))
        if entry.get('form'):
            print("Form URIs:", entry.get('form'))

        if entry.get('loadedSenses'):
            for sense in entry.get('loadedSenses', []):
                print("\n  --- Sentido ---")
                print("  Definición:", sense.get('definition') or "(no disponible)")
                if sense.get('language'):
                    print("  Idioma:", sense.get('language'))
                if sense.get('writtenReps'):
                    written = sense.get('writtenReps')
                    if isinstance(written, list):
                        written_strs = []
                        for w in written:
                            if isinstance(w, dict):
                                written_strs.append(w.get('uri') or w.get('@id') or str(w))
                            else:
                                written_strs.append(str(w))
                        written = "; ".join(written_strs)
                    print("  Etiquetas:", written)

                for rel_name in ["loaded_translation", "loaded_synonym", "loaded_antonym"]:
                    rel_items = sense.get(rel_name) or []
                    if rel_items:
                        label = rel_name.replace("loaded_", "").capitalize()
                        print(f"  {label}:")
                        for rel in rel_items:
                            rel_label = rel.get('writtenReps') or rel.get('@id') or rel.get('entryId')
                            if isinstance(rel_label, list):
                                rel_label_strs = []
                                for rl in rel_label:
                                    if isinstance(rl, dict):
                                        rel_label_strs.append(rl.get('uri') or rl.get('@id') or str(rl))
                                    else:
                                        rel_label_strs.append(str(rl))
                                rel_label = "; ".join(rel_label_strs)
                            print(f"    - {rel_label} ({rel.get('language')})")
        else:
            print("No se cargaron sentidos relacionados para esta entrada.")

    results = api_request(f"/ontologies/{acronym}/terminological_entries", params={"q": search_term, "page": 1, "pageSize": 5})

    print(f"Resultados de búsqueda para '{search_term}' en '{acronym}': {results.get('totalCount')} entradas encontradas.")
    for entry_summary in results.get('collection', [])[:5]:
        print("\n=== Entrada léxica ===")
        print("Label:", pretty_entry_label(entry_summary))
        entry_id = entry_summary.get('@id') or entry_summary.get('id')
        if entry_id:
            entry_details = get_terminological_entry(acronym, entry_id)
            print_lexical_entry(entry_details)
        else:
            print("No se encontró el ID de la entrada.")
else:
    print("No hay ontologías disponibles para buscar.")

Resultados de búsqueda para 'codi' en 'DM': 7 entradas encontradas.

=== Entrada léxica ===
Label: codi bidimensional
ID: http://myexample.com/terminologia_basica_dels_dispositius_mobils/terminologia_basica_dels_dispositius_mobils_C24_ca_codi_bidimensional_noun_entry
Texto: http://myexample.com/terminologia_basica_dels_dispositius_mobils/terminologia_basica_dels_dispositius_mobils_C24_ca_codi_bidimensional_noun_masculine_form
Idioma: {'uri': 'http://lexvo.org/id/iso639-3/cat'}
Form URIs: [{'uri': 'http://myexample.com/terminologia_basica_dels_dispositius_mobils/terminologia_basica_dels_dispositius_mobils_C24_ca_codi_bidimensional_noun_masculine_form'}]

  --- Sentido ---
  Definición: (no disponible)
  Translation:
    - 2-D bar code (http://lexvo.org/id/iso639-3/eng)
    - 2D barcode (http://lexvo.org/id/iso639-3/eng)
    - 2-dimensional bar code (http://lexvo.org/id/iso639-3/eng)
    - two-dimensional bar code (http://lexvo.org/id/iso639-3/eng)
    - two-dimensional barcode (http://l

## 6. Obtener Todos los Términos Léxicos con Traducciones y Sinónimos

Para una ontología léxica, se pueden descargar todos los términos de forma paginada, obteniendo para cada uno sus definiciones, traducciones y sinónimos relacionados.

In [38]:
# Obtener todos los términos léxicos con paginación, mostrando solo labels de traducciones y sinónimos
if ontologies:
    acronym = ontologies[0]['acronym']
    page_size = 10  # Términos por página
    page = 1
    
    # Primera llamada para obtener el total
    first_page = api_request(f"/ontologies/{acronym}/terminological_entries",
                            params={"page": page, "pageSize": page_size})
    total_terms = first_page.get('totalCount', 0)
    print(f"📊 Total de términos léxicos en '{acronym}': {total_terms}\n")
    print("=" * 80)
    
    # Procesar todas las páginas
    while True:
        results = api_request(f"/ontologies/{acronym}/terminological_entries",
                             params={"page": page, "pageSize": page_size})
        entries = results.get('collection', [])
        if not entries:
            break
        
        print(f"\n--- Página {page} (términos {(page-1)*page_size + 1} a {min(page*page_size, total_terms)}) ---\n")
        
        for entry_summary in entries:
            # Obtener el label del término
            term_label = pretty_entry_label(entry_summary)
            entry_id = entry_summary.get('@id') or entry_summary.get('id')
            
            print(f"📌 Término: {term_label}")
            
            if entry_id:
                try:
                    # Cargar detalles del término (con sus sentidos)
                    entry_details = get_terminological_entry(acronym, entry_id)
                    
                    # Procesar sentidos y sus relaciones (traducciones y sinónimos)
                    if entry_details.get('loadedSenses'):
                        has_relations = False
                        for sense in entry_details.get('loadedSenses', []):
                            # Traducciones
                            if sense.get('loaded_translation'):
                                translations = []
                                for trans in sense.get('loaded_translation', []):
                                    trans_label = trans.get('writtenReps') or trans.get('@id') or trans.get('entryId')
                                    if isinstance(trans_label, list):
                                        trans_label = "; ".join([str(x) if isinstance(x, str) else str(x) for x in trans_label])
                                    translations.append(str(trans_label))
                                if translations:
                                    print(f"  🌐 Traducciones: {', '.join(translations)}")
                                    has_relations = True
                            
                            # Sinónimos
                            if sense.get('loaded_synonym'):
                                synonyms = []
                                for syn in sense.get('loaded_synonym', []):
                                    syn_label = syn.get('writtenReps') or syn.get('@id') or syn.get('entryId')
                                    if isinstance(syn_label, list):
                                        syn_label = "; ".join([str(x) if isinstance(x, str) else str(x) for x in syn_label])
                                    synonyms.append(str(syn_label))
                                if synonyms:
                                    print(f"  ≈ Sinónimos: {', '.join(synonyms)}")
                                    has_relations = True
                        
                        if not has_relations:
                            print("  (sin traducciones ni sinónimos)")
                    else:
                        print("  (sin sentidos cargados)")
                        
                except Exception as e:
                    print(f"  ⚠️ Error al cargar detalles: {str(e)[:100]}")
            
            print()
        
        page += 1
        
        # Indicador de progreso
        current_count = min(page * page_size, total_terms)
        if current_count >= total_terms:
            print(f"✓ Procesados todos los {total_terms} términos.")
            break
        else:
            print(f"⏳ Procesados {current_count} de {total_terms} términos. Continuando...\n")
            print("=" * 80)
else:
    print("No hay ontologías disponibles.")

📊 Total de términos léxicos en 'DM': 651


--- Página 1 (términos 1 a 10) ---

📌 Término: 2-D bar code
  🌐 Traducciones: codi bidimensional, codi de barres bidimensional, codi de barres en 2D, codi en 2D, codi QR, código de barras 2D, código de barras bidimensional, code à barres 2D, code à barres à deux dimensions, code à barres bidimensionnel, code bidimensionnel
  ≈ Sinónimos: 2D barcode, 2-dimensional bar code, two-dimensional bar code, two-dimensional barcode

📌 Término: 2-dimensional bar code
  🌐 Traducciones: codi bidimensional, codi de barres bidimensional, codi de barres en 2D, codi en 2D, codi QR, código de barras 2D, código de barras bidimensional, code à barres 2D, code à barres à deux dimensions, code à barres bidimensionnel, code bidimensionnel
  ≈ Sinónimos: 2-D bar code, 2D barcode, two-dimensional bar code, two-dimensional barcode

📌 Término: 2D barcode
  🌐 Traducciones: codi bidimensional, codi de barres bidimensional, codi de barres en 2D, codi en 2D, codi QR, código

KeyboardInterrupt: 

## Conclusión

Este notebook ha demostrado las principales capacidades de la API de OntoPortal:

- **Conexión y configuración**: Cómo establecer la conexión con el servicio.
- **Exploración de ontologías**: Listar y obtener metadatos de vocabularios disponibles.
- **Búsqueda de conceptos**: Encontrar clases y términos dentro de ontologías.
- **Relaciones semánticas**: Obtener conceptos relacionados y jerarquías.
- **Consultas avanzadas**: Manejo de paginación, errores y parámetros complejos.

La API de OntoPortal ofrece una interfaz poderosa para trabajar con ontologías biomédicas, facilitando la integración en aplicaciones de análisis de datos, anotación automática y exploración de conocimiento semántico.

Para más información, consulta la documentación completa de la API en el repositorio del proyecto.